### ETL in Databricks
This notebook gives some basic commands for performing necessary ETL functions. Use this to create a medallion ETL pipeline using the financial dataset.

You should have:
 - bronze schema with tables for each set
 - silver schema with cleaned and formatted tables
 - gold schema with aggregated tables (to answer the questions in the notion page)

The notebook will be used as the source a daily job to refresh the pipeline (The whole notebook will be executed) and a dashboard will be created using the gold tables as source data.


In [0]:
%sql
-- Setup schemas for medallion architecture, you can also use the GUI
-- Schema == Database
CREATE SCHEMA IF NOT EXISTS jarvis_training.bronze;
CREATE SCHEMA IF NOT EXISTS jarvis_training.silver;
CREATE SCHEMA IF NOT EXISTS jarvis_training.gold;

In [0]:
# Insert database credentials and URL
username = dbutils.secrets.get(scope="jdbc", key="username")
password = dbutils.secrets.get(scope="jdbc", key="password")

url = "jdbc:sqlserver://ticdatabricks.database.windows.net:1433;databaseName=tic;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

transactions_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", username)
    .option("password", password)
    .load()
    ) 

display(transactions_df .limit(10))# display first 10 rows

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null
7475333,2010-01-01 00:07:00,1807,165,$4.81,Swipe Transaction,20519,Bronx,NY,10464.0,5942,null
7475334,2010-01-01 00:09:00,1556,2972,$77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475335,2010-01-01 00:14:00,1684,2140,$26.46,Online Transaction,39021,ONLINE,null,null,4784,null
7475336,2010-01-01 00:21:00,335,5131,$261.58,Online Transaction,50292,ONLINE,null,null,7801,null
7475337,2010-01-01 00:21:00,351,1112,$10.74,Swipe Transaction,3864,Flushing,NY,11355.0,5813,null


In [0]:
cards_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.cards_data")
    .option("user", username)
    .option("password", password)
    .load()
)
display(cards_df.limit(10))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No
4537,1746,Visa,Credit,4404898874682993,09/2003,736,YES,1,$27500,09/2003,2012,No
1278,1746,Visa,Debit,4001482973848631,07/2022,972,YES,2,$28508,02/2011,2011,No
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,YES,2,$9022,07/2003,2015,No
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,YES,2,$54,06/2010,2015,No
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,YES,1,$99,07/2006,2012,No


In [0]:
users_df = (spark.read
    .format("csv")
    .option("header", "true")
    .load("abfss://data@tic1.dfs.core.windows.net/users_data.csv")
)

display(users_df.limit(10))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
# Read the JSON file from volume
import json

file_path = "/Volumes/jarvis_training/bronze/raw_json/mcc_codes.json"

with open(file_path, "r", encoding="utf-8-sig") as f:
    mcc_dict = json.load(f)

mcc_rows = [(k, v) for k, v in mcc_dict.items()]

mcc_df = spark.createDataFrame(mcc_rows, ["mcc_code", "mcc_description"])

display(mcc_df.limit(10))

mcc_code,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees
4900,"Utilities - Electric, Gas, Water, Sanitary"
5942,Book Stores
5814,Fast Food Restaurants
4829,Money Transfer
5311,Department Stores


In [0]:
from pyspark.sql.functions import col, explode, map_entries, from_json, when
from pyspark.sql.types import StructType, StructField, MapType, StringType

fraud_text = (
    spark.read
    .option("wholetext", "true")
    .text("/Volumes/jarvis_training/bronze/raw_json/train_fraud_labels.json")
)

fraud_schema = StructType([
    StructField("target", MapType(StringType(), StringType()), True)
])

fraud_df = (
    fraud_text
    .select(from_json(col("value"), fraud_schema).alias("json_data"))
    .select(explode(map_entries(col("json_data.target"))).alias("entry"))
    .select(
        col("entry.key").alias("transaction_id"),
        when(col("entry.value") == "Yes", True)
        .when(col("entry.value") == "No", False)
        .otherwise(None)
        .alias("is_fraud")
    )
)

display(fraud_df.limit(10))

transaction_id,is_fraud
10649266,false
23410063,false
9316588,false
12478022,false
9558530,false
12532830,false
19526714,false
9906964,false
13224888,false
13749094,false


In [0]:
# Write your raw data to tables
transactions_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.transactions_data_bronze")

cards_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.cards_data_bronze")

users_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.users_data_bronze")
                                             
mcc_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.mcc_codes_bronze")

fraud_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.train_fraud_labels_bronze")